## Claude Sonnet 4 + Extended thinking

In [18]:
import datetime
import time
import pandas as pd
# To view tables in Google Colab
from google.colab import data_table
data_table.enable_dataframe_formatter()

# === API KEY SETUP ===
# Option 1: Retrieve API keys from your saved user data in Colab (safer method)
from google.colab import userdata
api_key = userdata.get('API_KEY')      # Key to access Claude AI
serpapi_key = userdata.get('SERPAPI_KEY')  # Key for search engine (not used in this example)

# Option 2: If you don't use userdata, you can directly insert your key
# api_key = "insert-your-API-key-here"

# Instructions for setting up API key in Google Colab:
# 1. Click on the key icon in the left sidebar
# 2. Add a new secret with name 'API_KEY'
# 3. Paste your Anthropic API key as the value

# Option 3: Loading from environment variables
# import os
# api_key = os.environ.get('ANTHROPIC_API_KEY')

# Define which Claude versions to use
CLAUDE_3_7 = "claude-3-7-sonnet-20250219"  # Claude 3.7 version
CLAUDE_4 = "claude-sonnet-4-20250514"     # Claude 4 version
models=[CLAUDE_3_7,CLAUDE_4]

In [ ]:
!pip install anthropic

In [19]:
from anthropic import Anthropic
client = Anthropic(
    api_key=api_key # API_KEY
)

In [20]:
def calculate_duration(end_time, start_time):
    '''Function to calculate inference duration'''
    duration = end_time - start_time
    return duration

In [21]:
#dataframe to collect the results
df_results = []

#SOME TEST PROMPTS

induction_question = "Prove by induction that the sum of the first n natural numbers is n*(n+1)/2. Display the result in plain text."
powers_question = "Prove that for every natural number n ≥ 0, the following formula is true: sum(2^i, i=0..n) = 2^(n+1) - 1. Display the result in plain text."
prime_numbers = "Are there infinite prime numbers such that n mod 4 == 3?"

# Reasoning questions
business_analysis = """You are the head of innovation at a medium-sized manufacturing company that has historically been resistant to change. The company is profitable but is gradually losing market share to more innovative competitors. Your CEO is pragmatic and cost-conscious, and considers innovation as a risk rather than an opportunity.
                    Develop a comprehensive strategy to convince your manager to invest in innovation, considering:
                    1. How to present a compelling business case that demonstrates the ROI of innovation
                    2. What data and metrics to use to support your argument
                    3. A gradual implementation proposal that minimizes risks
                    4. How to address potential objections based on past failures of change initiatives
                    5. A communication plan to gain support from employees and other stakeholders.
                    Include concrete examples of similar companies that have successfully implemented innovation programs and the results they achieved."""

# Simple test questions
strawberry_question = "How many 'r's are there in the word 'strawberry'?"
logic_question = "Alice has one brother and 2 sisters, how many sisters does Alice's brother have?"

prompts=[
      ('prime_numbers', prime_numbers),
      ('logic_puzzle', logic_question),
      ('business_analysis', business_analysis),
      ('powers_question', powers_question),
      ('induction_question', induction_question),
      ('strawberry_question', strawberry_question)
    ]


## CLAUDE + EXTENDED THINKING

In [22]:
def prompt_claude(prompt, model_name):
    """
    Function that asks a question to a specific Claude model

    Parameters:
    - prompt: the prompt or question to Claude
    - model_name: which version of Claude to use

    Returns:
    - time: how long Claude took
    - thinking: the thinking process
    - response: the final response
    """
    print(f"Prompt for {model_name}: {prompt}")

    # Start measuring time
    start_time = time.time()

    # Call Claude with thinking enabled
    response = client.messages.create(
        model=model_name,
        max_tokens=16000,
        thinking={
            "type": "enabled",
            "budget_tokens": 10000
        },
        messages=[{
            "role": "user",
            "content": prompt #choose one of the above prompts
        }]
    )

    # Calculate how much time has passed
    end_time = time.time()
    duration = end_time - start_time

    # Extract thinking and response
    thinking = None
    response_text = None

    for block in response.content:
        if block.type == "thinking":
            thinking = block.thinking
            print(f"\nThinking summary: {block.thinking}")
        elif block.type == "text":
            response_text = block.text
            print(f"\nResponse: {block.text}")
    return duration, thinking, response_text

In [ ]:
#client code: Claude 3.7

duration, thinking, response = prompt_claude(logic_question, CLAUDE_3_7)
print(duration, thinking, response)


In [ ]:
#client code: Claude 4
#duration, thinking, response = prompt_claude(logic_question, CLAUDE_4)
#print(duration, thinking, response)

# Extended Thinking Mode: Comparing Claude Models


In [ ]:
# Evaluating inference speed differences between 3.7 and 4
for m in models:

  for prompt_name, p in prompts:
    time_taken, thinking, response = prompt_claude(p, m)

    print(f"Time: {time_taken:.2f} seconds")
    print(f"\n{m}'s thinking:")
    print("------------------------")
    print(thinking)
    print("------------------------")
    print(f"\n{m}'s response:")
    print("------------------------")
    print(response)
    print("------------------------")

      # Save results
    df_results.append({
        "Prompt": prompt_name,
        "Model": m,
        "Time": time_taken,
        "Thinking length (characters)": len(thinking) if thinking else 0,
        "Response length (characters)": len(response) if response else 0
    })

    # Small pause to not overload the API
    time.sleep(1)



In [ ]:
# === CREATE A SUMMARY TABLE ===
table = pd.DataFrame( df_results)
table

In [ ]:
import matplotlib.pyplot as plt

# Pivot the table to have prompts as rows and models as columns for easier plotting
pivot_table = table.pivot_table(index='Prompt', columns='Model', values='Time')

# Create the bar chart
pivot_table.plot(kind='bar', figsize=(12, 6))

plt.title('Comparison of Inference Time per Prompt for Claude Models')
plt.xlabel('Prompt')
plt.ylabel('Time (seconds)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Model')
plt.tight_layout() # Adjust layout to prevent labels overlapping
plt.show()

****
